# A FAIR² Mental Health Survey Dataset Exploration with `mlcroissant`
This notebook demonstrates loading, overview, extraction, processing, and visualization of the survey dataset from Kilifi County, Kenya, using the `mlcroissant` library. All entities are referenced by their `@id`.

### Dataset Source
The dataset source is provided via a Croissant schema URL:

`https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json`

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlcimport pandas as pdimport matplotlib.pyplot as pltimport numpy as np# Define the dataset URLcroissant_url = 'https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json'# Load the dataset metadatadataset = mlc.Dataset(croissant_url)metadata = dataset.metadata.to_json()print(f"Dataset title: {metadata['name']}\nDescription: {metadata['description']}")print(f"Published on: {metadata['datePublished']}")print(f"License: {metadata['license']}")print(f"Available record sets: {metadata['recordSet']}")

## 2. Data Overview
Review available record sets and their IDs. Explore included fields and columns for each record set.

In [ ]:
# List record set IDs (referenced by @id)record_set_ids = dataset.metadata.to_json()['recordSet']# If record sets are objects, fetch their details (fields, columns)for record_set_id in record_set_ids:    print(f"\nRecord Set @id: {record_set_id}")    rset = dataset.record_set(record_set_id)    fields = rset.fields    print("Fields and Their @ids:")    for field in fields:        print(f"  {field['@id']} (name: {field.get('name', field['@id'])}, type: {field.get('dataType','unknown')})")    columns = rset.columns    if columns:        print("Columns and Their @ids:")        for col in columns:            print(f"  {col['@id']} (name: {col.get('name', col['@id'])}, type: {col.get('dataType','unknown')})")

## 3. Data Extraction
Load data from each record set into a DataFrame for analysis. All record set and field references use their `@id`.

In [ ]:
# Collect all record set @ids and load their records into DataFramesrecord_sets = dataset.metadata.to_json()['recordSet']dataframes = {}print("\nExtracting records:")for rs_id in record_sets:    records = list(dataset.records(record_set=rs_id))    df = pd.DataFrame(records)    dataframes[rs_id] = df    print(f"Record Set @id: {rs_id}")    print(f"Columns: {df.columns.tolist()}")    print(df.head(2), '\n')# Choose a main record set (assume first is survey responses)main_record_set_id = record_sets[0]print(f"\nUsing main record set: {main_record_set_id}")print("Columns:", dataframes[main_record_set_id].columns.tolist())dataframes[main_record_set_id].head()

## 4. Exploratory Data Analysis (EDA)
Filter, normalize, group, and analyze numeric and categorical fields using their `@id`.
We'll analyze GAD-7 scores, referenced by its `@id`.

In [ ]:
# Identify numeric and group fields by their @id (example: GAD-7 score and gender)df = dataframes[main_record_set_id]# Find columns matching possible survey score fieldsprint("Available numeric score fields:")numeric_fields = []for col in df.columns:    if "gad7" in col.lower() or "phq9" in col.lower() or "psq" in col.lower() or "score" in col.lower():        print(f"  {col}")        numeric_fields.append(col)# Use GAD-7 score column as example (replace with exact @id if available)gad7_score_id = numeric_fields[0] if numeric_fields else df.columns[0]# Also choose a group field (e.g. gender, referenced by @id)group_fields = [c for c in df.columns if 'gender' in c.lower() or 'sex' in c.lower()]group_field_id = group_fields[0] if group_fields else None# Filter for GAD-7 score > 10 (moderate to severe anxiety)threshold = 10filtered_df = df[df[gad7_score_id] > threshold]print(f"Filtered records with {gad7_score_id} > {threshold}:")print(filtered_df.head())filtered_df[f"{gad7_score_id}_normalized"] = (filtered_df[gad7_score_id] - filtered_df[gad7_score_id].mean()) / filtered_df[gad7_score_id].std()print(f"Normalized {gad7_score_id} for filtered records:")print(filtered_df[[gad7_score_id, f"{gad7_score_id}_normalized"]].head())# Group by gender (or similar demographic)if group_field_id:    grouped_df = filtered_df.groupby(group_field_id)[gad7_score_id].mean().reset_index()    print(f"Grouped mean {gad7_score_id} by {group_field_id}:\n", grouped_df.head())

## 5. Visualization
Visualize numeric score distributions and relationships between demographic and survey results. Use field and group `@id`s.

In [ ]:
# Plot histogram of GAD-7 scoresplt.figure(figsize=(8,4))df[gad7_score_id].dropna().hist(bins=20)plt.title(f"Distribution of GAD-7 Scores ({gad7_score_id})")plt.xlabel("GAD-7 Score")plt.ylabel("Frequency")plt.show()# Boxplot by demographic group if availableif group_field_id:    plt.figure(figsize=(8,4))    df.boxplot(column=gad7_score_id, by=group_field_id)    plt.title(f"{gad7_score_id} by {group_field_id}")    plt.suptitle("")    plt.xlabel(group_field_id)    plt.ylabel("GAD-7 Score")    plt.show()

## 6. Conclusion
This notebook loaded and explored the Kilifi Mental Health Survey dataset using `mlcroissant`, referencing all entities by their `@id`. We reviewed record sets, extracted survey responses, filtered and normalized GAD-7 scores, grouped by demographic fields (such as gender), and visualized distributions.

Key findings:
* GAD-7 score distributions highlight moderate and severe anxiety levels in subsets of respondents.
* Grouping by demographic fields (e.g., gender) reveals potential disparities.
* The Croissant schema and `mlcroissant` enable AI-ready and reproducible data analysis.